In [4]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split
import pickle
import tensorflow
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import TensorBoard, EarlyStopping
import datetime


data = pd.read_csv("Churn_Modelling.csv")
data = data.drop(["RowNumber", "CustomerId", "Surname"], axis=1)

LabelEncoder_gender = LabelEncoder()
data['Gender'] = LabelEncoder_gender.fit_transform(data['Gender'])

OneHotEncoder_geography = OneHotEncoder()
geo_encoded = OneHotEncoder_geography.fit_transform(data[['Geography']]).toarray()
geo_names = OneHotEncoder_geography.get_feature_names_out(['Geography'])
geo_df = pd.DataFrame(geo_encoded, columns=geo_names)
data = pd.concat([data.drop('Geography', axis = 1), geo_df], axis=1)

X = data.drop('Exited', axis=1)
Y = data['Exited']

X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

with open('gender.pkl', 'wb') as file:
    pickle.dump(LabelEncoder_gender, file)

with open('geography.pkl', 'wb') as file:
    pickle.dump(OneHotEncoder_geography, file)

with open('scaler.pkl', 'wb') as file:
    pickle.dump(scaler, file)

seq_model = Sequential([
    Dense(64, activation ='relu', input_shape=(X_train.shape[1],)),
    Dense(32, activation='relu'),
    Dense(1, activation='sigmoid')
])

_opt = tensorflow.keras.optimizers.Adam(learning_rate=0.001)
_loss = tensorflow.keras.losses.BinaryCrossentropy()
_metrics = tensorflow.keras.metrics.Accuracy()

seq_model.compile(optimizer=_opt, loss=_loss, metrics=[_metrics])

log_dir = 'logs5/fit/'+datetime.datetime.now().strftime('%d-%m-%Y_%H-%M-%S')

tensorflow_callback = TensorBoard(log_dir=log_dir, histogram_freq=1)
earlystopping_callback = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

seq_model.fit(X_train, Y_train, validation_data=(X_test, Y_test), epochs=10, callbacks=[tensorflow_callback, earlystopping_callback])


%reload_ext tensorboard

%tensorboard --logdir logs5/fit

c:\Users\bdn7kor\Documents\Projects\Courses\ANN\env2\Lib\site-packages\keras\src\layers\core\dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.0000e+00 - loss: 0.4494 - val_accuracy: 0.0000e+00 - val_loss: 0.3936
Epoch 2/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.0000e+00 - loss: 0.3897 - val_accuracy: 0.0000e+00 - val_loss: 0.3596
Epoch 3/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.0000e+00 - loss: 0.3584 - val_accuracy: 0.0000e+00 - val_loss: 0.3520
Epoch 4/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.0000e+00 - loss: 0.3437 - val_accuracy: 0.0000e+00 - val_loss: 0.3445
Epoch 5/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.0000e+00 - loss: 0.3374 - val_accuracy: 0.0000e+00 - val_loss: 0.3427
Epoch 6/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.0000e+00 - loss: 0.3338 - val_accuracy: 0.0000e+00 - val_loss: 0.3390
Epoch 7/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.0000e+00 - loss: 0.3312 - val_accuracy: 0.0000e+00 - val_loss: 0.3400
Epoch 8/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/s

Reusing TensorBoard on port 6006 (pid 25752), started 8:46:22 ago. (Use '!kill 25752' to kill it.)

In [5]:
%reload_ext tensorboard
%tensorboard --logdir logs5/fit

Reusing TensorBoard on port 6006 (pid 25752), started 8:46:22 ago. (Use '!kill 25752' to kill it.)

In [6]:

seq_model.save('churn_model.keras')

model_loaded = tensorflow.keras.models.load_model('churn_model.keras')

with open('gender.pkl', 'rb') as file:
    LabelEncoder_gender = pickle.load(file)

with open('geography.pkl', 'rb') as file:
    OneHotEncoder_geography = pickle.load(file)

with open('scaler.pkl', 'rb') as file:
    scaler = pickle.load(file)

In [ ]:
import pandas as pd

input_data = {
    'CreditScore': 600,
    'Geography': 'France',
    'Gender': 'Male',
    'Age': 40,
    'Tenure': 3,
    'Balance': 60000,
    'NumOfProducts': 2,
    'HasCrCard': 1,
    'IsActiveMember': 1,
    'EstimatedSalary': 50000
}

input_data = pd.DataFrame([input_data])

#Gender
input_data['Gender'] = LabelEncoder_gender.transform(input_data['Gender'])

#Geography
geo_encoded = OneHotEncoder_geography.transform(input_data[['Geography']]).toarray()

input_df = pd.DataFrame(geo_encoded, columns=OneHotEncoder_geography.get_feature_names_out(['Geography']))


input_data = pd.concat([input_data.drop('Geography', axis=1), input_df], axis=1)

#scaler

input_data_scaled = scaler.transform(input_data)

#prediction
prediction = seq_model.predict(input_data_scaled)

#prediction probability
prediction[0][0]

if prediction[0][0] > 0.5:
    print(f'Churn Probability: {prediction[0][0]:.2f} - Likely to Churn')
    print("The customer is likely to churn.")
else:
    print(f'Churn Probability: {prediction[0][0]:.2f} - Unlikely to Churn')
    print("The customer is unlikely to churn.")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
Churn Probability: 0.03 - Unlikely to Churn
The customer is unlikely to churn.


In [8]:
from tensorflow.keras.models import load_model

seq_model.save('churn_model_v2.keras')

loaded_model = load_model('churn_model_v2.keras')

with open('gender.pkl', 'rb') as file:
    pickle.load(file)

with open('geography.pkl', 'rb') as file:
    pickle.load(file)

with open('scaler.pkl', 'rb') as file:
    pickle.load(file)


In [24]:
input_data_v2 = {
    'CreditScore': 600,
    'Geography': 'Germany', 
    'Gender': 'Female',
    'Age': 29,
    'Tenure': 3,
    'Balance': 60,
    'NumOfProducts': 2,
    'HasCrCard': 1,
    'IsActiveMember': 1,
    'EstimatedSalary': 50000
}

In [25]:
input_data_df = pd.DataFrame([input_data_v2])

#gender
input_data_df['Gender'] = LabelEncoder_gender.transform(input_data_df['Gender'])

#geography
geo_encoded = OneHotEncoder_geography.transform(input_data_df[['Geography']]).toarray()

geo_names = OneHotEncoder_geography.get_feature_names_out(['Geography'])

geo_df = pd.DataFrame(geo_encoded, columns=geo_names)

input_data_df = pd.concat([input_data_df.drop('Geography', axis = 1), geo_df], axis=1)

input_data_scaled_v2 = scaler.transform(input_data_df)

#prediction
prediction = loaded_model.predict(input_data_scaled_v2)

#prediction probability
prediction_probability = prediction[0][0]

if prediction_probability > 0.5:
    print(f"The customer is likely to churn with a probability of {prediction_probability:.2f}.")
else:
    print(f"The customer is unlikely to churn with a probability of {prediction_probability:.2f}.")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step
The customer is unlikely to churn with a probability of 0.02.
